In [ ]:
import os
import json
import gc
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
IMAGE_DIR    = "./images"          # folder containing raw images
OUTPUT_DIR   = "./sam_output"      # where per-image JSON will be saved
SAM_WEIGHTS  = "./sam_weights/sam_vit_h_4b8939.pth"
MODEL_TYPE   = "vit_h"             # must match the weight file

# ─── MASK GENERATOR PARAMETERS ────────────────────────────────────────────────
# Tune these for small, densely packed rock objects:
SAM_PARAMS = dict(
    points_per_side          = 64,    # denser grid → more small objects detected
                                      # default=32; increase for tiny rocks
    points_per_batch         = 64,    # how many points fed to SAM at once
                                      # lower if VRAM is tight (try 32 or 16)
    pred_iou_thresh          = 0.80,  # predicted IoU quality threshold
                                      # lower → more masks kept (noisier)
    stability_score_thresh   = 0.90,  # mask boundary stability threshold
    stability_score_offset   = 1.0,
    box_nms_thresh           = 0.70,  # NMS overlap threshold between masks
    crop_n_layers            = 1,     # multi-crop pass for small objects
                                      # 0=disabled, 1=one extra crop pass
    crop_nms_thresh          = 0.70,
    crop_overlap_ratio       = 512/1500,
    crop_n_points_downscale_factor = 2,
    min_mask_region_area     = 200,   # pixels² — drop masks smaller than this
                                      # tune based on your smallest rock size
)

# ─── POST-FILTER ──────────────────────────────────────────────────────────────
MIN_AREA_PX  = 200    # absolute minimum area in pixels (redundant safety check)
MAX_AREA_PX  = None   # set a value to exclude full-image background masks
                      # e.g., 0.5 * image_width * image_height

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Images   : {IMAGE_DIR}")
print(f"Output   : {OUTPUT_DIR}")
print(f"Weights  : {SAM_WEIGHTS}")
print(f"CUDA     : {torch.cuda.is_available()}")

In [ ]:
def load_sam_model(weights_path: str, model_type: str, device: str = "cuda"):
    """Load SAM model onto the specified device."""
    print(f"Loading SAM ({model_type}) from {weights_path} ...")
    sam = sam_model_registry[model_type](checkpoint=weights_path)
    sam.to(device=device)
    sam.eval()
    print("SAM loaded successfully.")
    return sam

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sam_model = load_sam_model(SAM_WEIGHTS, MODEL_TYPE, DEVICE)

In [ ]:
def mask_to_polygons(binary_mask: np.ndarray, simplify_tolerance: float = 1.5):
    """
    Convert a binary mask (H×W uint8) to a list of polygons.
    Each polygon is a flat list [x1,y1, x2,y2, ...] for CVAT compatibility.

    Returns a list of polygons (one mask can have multiple contours
    if it contains holes or disconnected regions — we keep only the
    largest contour per mask to stay clean).
    """
    # Ensure uint8
    mask_uint8 = (binary_mask > 0).astype(np.uint8) * 255

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,      # only outer contours — ignores holes
        cv2.CHAIN_APPROX_SIMPLE # compress straight edges
    )

    if not contours:
        return []

    # Keep only the largest contour (avoids tiny noise fragments)
    largest = max(contours, key=cv2.contourArea)

    # Douglas-Peucker simplification to reduce vertex count
    epsilon = simplify_tolerance
    approx = cv2.approxPolyDP(largest, epsilon, closed=True)

    # Need at least 3 points for a valid polygon
    if len(approx) < 3:
        return []

    # Flatten to [x1, y1, x2, y2, ...]
    points = approx.reshape(-1, 2).tolist()
    flat   = [coord for point in points for coord in point]
    return [flat]  # list of polygons (one per contour kept)


def get_image_files(image_dir: str):
    """Return sorted list of image paths from a directory."""
    supported = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}
    paths = sorted([
        p for p in Path(image_dir).iterdir()
        if p.suffix.lower() in supported
    ])
    return paths

In [ ]:
def process_single_image(
    image_path: Path,
    mask_generator: SamAutomaticMaskGenerator,
    min_area: int,
    max_area,
    simplify_tolerance: float = 1.5
):
    """
    Run SAM AutomaticMaskGenerator on a single image.

    Returns a dict with image metadata and list of mask records,
    each containing:
      - area (int)
      - bbox [x, y, w, h]
      - predicted_iou (float)
      - stability_score (float)
      - polygons: list of flat point lists
    """
    # ─ Read image ────────────────────────────────────────────────────────────
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        print(f"  [WARN] Could not read {image_path.name}, skipping.")
        return None

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]

    # ─ Generate masks ─────────────────────────────────────────────────────────
    masks = mask_generator.generate(rgb)

    # ─ Filter ─────────────────────────────────────────────────────────────────
    image_area = h * w
    filtered = []
    for m in masks:
        area = int(m["area"])
        if area < min_area:
            continue
        if max_area is not None and area > max_area:
            continue
        filtered.append(m)

    # ─ Convert to polygons ────────────────────────────────────────────────────
    records = []
    for m in filtered:
        polys = mask_to_polygons(
            m["segmentation"].astype(np.uint8),
            simplify_tolerance=simplify_tolerance
        )
        if not polys:
            continue

        records.append({
            "area"            : int(m["area"]),
            "bbox"            : [int(v) for v in m["bbox"]],  # [x,y,w,h]
            "predicted_iou"   : float(m["predicted_iou"]),
            "stability_score" : float(m["stability_score"]),
            "polygons"        : polys   # [ [x1,y1,x2,y2,...], ... ]
        })

    return {
        "filename" : image_path.name,
        "width"    : w,
        "height"   : h,
        "n_masks"  : len(records),
        "masks"    : records
    }


def run_pipeline(
    image_dir    : str,
    output_dir   : str,
    sam_model,
    sam_params   : dict,
    min_area     : int,
    max_area,
    skip_existing: bool = True,
    simplify_tol : float = 1.5
):
    """
    Process all images in image_dir sequentially.

    Memory strategy:
      - MaskGenerator is constructed once and reused.
      - After each image: torch.cuda.empty_cache() + gc.collect()
        to release intermediate GPU tensors.
      - Images are loaded one at a time (no batching needed — SAM
        handles its own internal batching via points_per_batch).
    """
    image_paths = get_image_files(image_dir)
    print(f"Found {len(image_paths)} images in '{image_dir}'")

    # Build mask generator once
    mask_generator = SamAutomaticMaskGenerator(sam_model, **sam_params)

    results = {"processed": 0, "skipped": 0, "failed": 0}

    for img_path in tqdm(image_paths, desc="Processing images"):
        out_path = Path(output_dir) / (img_path.stem + ".json")

        # Skip already-processed images on re-runs
        if skip_existing and out_path.exists():
            results["skipped"] += 1
            continue

        try:
            record = process_single_image(
                img_path, mask_generator,
                min_area, max_area, simplify_tol
            )

            if record is None:
                results["failed"] += 1
                continue

            with open(out_path, "w") as f:
                json.dump(record, f)  # compact — no indent for large files

            results["processed"] += 1
            tqdm.write(
                f"  {img_path.name}: {record['n_masks']} masks "
                f"({record['width']}×{record['height']})"
            )

        except Exception as e:
            tqdm.write(f"  [ERROR] {img_path.name}: {e}")
            results["failed"] += 1

        finally:
            # ── Critical: free GPU memory after EVERY image ───────────────
            torch.cuda.empty_cache()
            gc.collect()

    print("\n──── Run Summary ────")
    print(f"  Processed : {results['processed']}")
    print(f"  Skipped   : {results['skipped']}  (already done)")
    print(f"  Failed    : {results['failed']}")
    return results


# ─── RUN ──────────────────────────────────────────────────────────────────────
run_pipeline(
    image_dir     = IMAGE_DIR,
    output_dir    = OUTPUT_DIR,
    sam_model     = sam_model,
    sam_params    = SAM_PARAMS,
    min_area      = MIN_AREA_PX,
    max_area      = MAX_AREA_PX,
    skip_existing = True,        # safe to re-run; won't reprocess done images
    simplify_tol  = 1.5          # increase to reduce polygon vertices further
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

def visualize_result(image_dir, output_dir, filename, max_display=150):
    """Overlay SAM polygons on the original image for verification."""
    img_path = Path(image_dir) / filename
    json_path = Path(output_dir) / (Path(filename).stem + ".json")

    bgr = cv2.imread(str(img_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    with open(json_path) as f:
        record = json.load(f)

    fig, ax = plt.subplots(1, 1, figsize=(16, 10))
    ax.imshow(rgb)

    patches = []
    for i, mask in enumerate(record["masks"][:max_display]):
        for poly_pts in mask["polygons"]:
            pts = np.array(poly_pts).reshape(-1, 2)
            patch = MplPolygon(pts, closed=True)
            patches.append(patch)

    collection = PatchCollection(
        patches,
        alpha=0.35,
        facecolor=np.random.rand(len(patches), 3),
        edgecolor="white",
        linewidth=0.5
    )
    ax.add_collection(collection)
    ax.set_title(f"{filename}  —  {record['n_masks']} masks detected", fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# Replace with one of your actual image filenames
visualize_result(IMAGE_DIR, OUTPUT_DIR, "img_001.jpg")